# Cosmic background systematic covariances (notebook production)

Build unisim covariances from **offbeam data** vs **intime MC** (gate-scaled), matching
``get_systematics_cosmics.py run-ana`` and ``syst_cosmics_common.build_variable_configs``.

- Loads cosmic template samples via ``files_config.get_ana_dfs(option='cosmics_systs')``.
- Loads **final-selection MC** like ``data_mc_comparison.ipynb`` for topology cosmic contamination.
- Scales template uncertainties by the per-bin contamination fraction (``get_topo_category``).
- Replaces per-bin cosmic template ``cov_frac`` with a **flat, uncorrelated** matrix: every bin gets
  the largest ``sqrt(diag(cov_frac))`` among bins that did not blow up (limited-stats spikes).
- Writes ``Cosmics/cosmics_syst_dict.npz`` plus ``cosmics_uncertainty_summary.json``.

For grid-scale I/O use ``syst_cosmics_chunk.py`` + ``syst_cosmics_aggregate.py`` (optional cell below).

In [19]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [20]:
from os import path, makedirs
from datetime import datetime
from pathlib import Path
import json
import os
import time
import warnings

import numpy as np
import pandas as pd
from pandas.errors import PerformanceWarning
from tqdm import tqdm

import sys
sys.path.append('/exp/sbnd/app/users/munjung/xsec/freeze/cafpyana')

from pyanalib.split_df_helpers_new import dfs_from_dir
from pyanalib.variable_calculator import get_cc1p0pi_tki
from pyanalib.pandas_helpers import pad_column_name

from analysis_village.numucc_1p0pi.variable_configs import VariableConfig
from makedf.util import InFV
from analysis_village.numucc_1p0pi.utils import (
    get_clipped_evts,
    plot_frac_unc,
    plot_heatmap,
)
from analysis_village.numucc_1p0pi.files_config import save_fig_base_dir
from analysis_village.numucc_1p0pi.categories import get_topo_category
from analysis_village.numucc_1p0pi.dataset_locations import MULTISIM_SYST_GLOBS_FINAL, SPRING_GEN1_ROOT
from analysis_village.numucc_1p0pi.syst_cosmics_common import build_variable_configs
from analysis_village.numucc_1p0pi.scripts.get_systematics_cosmics import (
    load_cosmic_samples_ana,
    process_variable_cosmics,
    save_cosmics_npz,
)
from analysis_village.numucc_1p0pi.syst_disk_layout import (
    FILE_COSMICS,
    SUB_COSMICS,
    category_out_dir,
    normalized_root,
)

warnings.filterwarnings('ignore', category=PerformanceWarning)
import matplotlib.pyplot as plt
plt.style.use('presentation.mplstyle')


def _ts():
    return datetime.now().strftime('%H:%M:%S')


def _log(msg):
    print(f'[{_ts()}] {msg}', flush=True)


def frac_unc_from_pack(pack):
    return np.sqrt(np.maximum(np.diag(pack['cov_frac']), 0.0))


def topo_cosmic_contamination_fraction(mc_df, var_config):
    """Per-bin weighted cosmic fraction in selected MC (``get_topo_category`` cut 0)."""
    cut_cosmic, *_ = get_topo_category(mc_df, ret_cuts=True)
    mc_cosmic = mc_df.loc[cut_cosmic]

    if var_config.var_save_name == 'integrated':
        if 'pot_weight' in mc_df.columns:
            n_total = np.array([float(mc_df['pot_weight'].sum())])
            n_cosmic = np.array([float(mc_cosmic['pot_weight'].sum())])
        else:
            n_total = np.array([float(len(mc_df))])
            n_cosmic = np.array([float(len(mc_cosmic))])
    else:
        var_all, w_all = get_clipped_evts(mc_df, var_config.var_evt_reco_col, var_config.bins)
        var_cos, w_cos = get_clipped_evts(mc_cosmic, var_config.var_evt_reco_col, var_config.bins)
        n_total, _ = np.histogram(var_all, bins=var_config.bins, weights=w_all)
        n_cosmic, _ = np.histogram(var_cos, bins=var_config.bins, weights=w_cos)

    frac = np.where(n_total > 0, n_cosmic / n_total, 0.0)
    return frac.astype(float), n_cosmic.astype(float), n_total.astype(float)


def scale_cov_frac_by_contamination(cov_frac, contam_frac):
    f = np.asarray(contam_frac, dtype=float)
    c = np.asarray(cov_frac, dtype=float)
    return c * np.outer(f, f)


def selected_rate_uncertainty_from_cosmics(cosmic_pay, contam_frac):
    """Propagate cosmic-template fractional uncertainty to selected event rate."""
    cov_template = np.asarray(cosmic_pay['rate']['cov_frac'], dtype=float)
    cov_selected = scale_cov_frac_by_contamination(cov_template, contam_frac)
    frac_unc_template = frac_unc_from_pack(cosmic_pay['rate'])
    frac_unc_selected = frac_unc_template * np.asarray(contam_frac, dtype=float)
    return {
        'cov_frac': cov_selected,
        'frac_unc': frac_unc_selected,
        'frac_unc_template': frac_unc_template,
        'contamination_fraction': np.asarray(contam_frac, dtype=float),
    }

In [21]:
# --- configuration (override via env) ---
CV_MODE = os.environ.get('COSMICS_CV_MODE', 'offbeam')  # offbeam | intime | mean
_vars_env = os.environ.get('COSMICS_VARS', '').strip()
VAR_NAMES = [x.strip() for x in _vars_env.split(',') if x.strip()] or None

_flat_env = os.environ.get('COSMICS_FLAT_UNC', '1').strip().lower()
COSMICS_FLAT_UNC = _flat_env not in ('0', 'false', 'no')
COSMICS_BLOWUP_FRAC_UNC = float(os.environ.get('COSMICS_BLOWUP_FRAC_UNC', '1.0'))

if CV_MODE not in ('offbeam', 'intime', 'mean'):
    raise ValueError(f'COSMICS_CV_MODE must be offbeam|intime|mean, got {CV_MODE!r}')

var_configs = build_variable_configs(VAR_NAMES)
_log(
    f'cv_mode={CV_MODE} flat_unc={COSMICS_FLAT_UNC} '
    f'blowup_thresh={COSMICS_BLOWUP_FRAC_UNC} n_variables={len(var_configs)}'
)
if VAR_NAMES:
    _log(f'  subset: {VAR_NAMES}')

[16:57:07] cv_mode=offbeam flat_unc=True blowup_thresh=1.0 n_variables=24


In [22]:
var_configs = [VariableConfig.all_events(),
                VariableConfig.muon_momentum(),
                VariableConfig.muon_direction(),
                VariableConfig.proton_momentum(),
                VariableConfig.proton_direction(),
                VariableConfig.tki_del_alpha(),
                VariableConfig.tki_del_phi(),
                VariableConfig.tki_del_Tp(),
                VariableConfig.tki_del_p(),
                VariableConfig.tki_del_Tp_x(),
                VariableConfig.tki_del_Tp_y(),
                VariableConfig.muon_direction_x(),
                VariableConfig.muon_direction_y(),
                VariableConfig.proton_direction_x(),
                VariableConfig.proton_direction_y(),
                # VariableConfig.opening_angle(),
                VariableConfig.vertex_x(),
                VariableConfig.vertex_y(),
                VariableConfig.vertex_z(),
                ]

In [23]:
today_str = datetime.now().strftime('%Y%m%d')
SYST_DISK_ROOT = path.join(save_fig_base_dir, f'systematics-notebook-cosmic-{today_str}')
cosmics_disk_dir = category_out_dir(SYST_DISK_ROOT, SUB_COSMICS)
makedirs(cosmics_disk_dir, exist_ok=True)
COSMICS_NPZ = path.join(cosmics_disk_dir, FILE_COSMICS)
COSMICS_SUMMARY_JSON = path.join(cosmics_disk_dir, 'cosmics_uncertainty_summary.json')

SAVE_PLOTS = False
PLOT_FRAC_UNC_EACH_VAR = True
PLOT_HEATMAP_FIRST_VAR = True

_log('syst disk Cosmics dir: ' + normalized_root(cosmics_disk_dir))

[16:57:07] syst disk Cosmics dir: /exp/sbnd/data/users/munjung/plots/numucc1p0pi/systematics-notebook-cosmic-20260519/Cosmics


In [24]:
_log('loading offbeam data + intime MC (cosmics_systs) ...')
t_load = time.time()
offbeam_df, intime_df = load_cosmic_samples_ana()
_log(
    f'  offbeam evt={len(offbeam_df):,} intime evt={len(intime_df):,} '
    f'in {time.time() - t_load:.1f}s'
)
if 'pot_scale' not in intime_df.columns:
    raise RuntimeError('intime MC missing pot_scale column from get_ana_dfs(cosmics_systs)')
_log(f'  intime pot_scale (unique): {intime_df["pot_scale"].unique()}')

[16:57:07] loading offbeam data + intime MC (cosmics_systs) ...
Reading file with tag aa, mc_n_split: 1
Reading file with tag ab, mc_n_split: 1
Reading file with tag ac, mc_n_split: 1
intime cosmics data gates: 8.34e+07
Reading file with tag aa, mc_n_split: 1
Reading file with tag ab, mc_n_split: 1
Reading file with tag ac, mc_n_split: 1
Reading file with tag ad, mc_n_split: 1
Reading file with tag ae, mc_n_split: 1
Reading file with tag af, mc_n_split: 1
Reading file with tag ag, mc_n_split: 1
Reading file with tag ah, mc_n_split: 1
Reading file with tag ai, mc_n_split: 1
Reading file with tag aj, mc_n_split: 1
Reading file with tag ak, mc_n_split: 1
Reading file with tag al, mc_n_split: 1
Reading file with tag am, mc_n_split: 1
Reading file with tag an, mc_n_split: 1
Reading file with tag ao, mc_n_split: 1
Reading file with tag ap, mc_n_split: 1
Reading file with tag aq, mc_n_split: 1
Reading file with tag ar, mc_n_split: 1
Reading file with tag as, mc_n_split: 1
Reading file with ta

## Final-selection MC (cosmic contamination)

Load scaled MC from ``MULTISIM_SYST_GLOBS_FINAL['MCstat']`` and apply the same per-TPC + FV + topology labeling as ``data_mc_comparison.ipynb``.

In [25]:
_mc_glob = MULTISIM_SYST_GLOBS_FINAL.get('MCstat')
if not _mc_glob:
    raise RuntimeError('MULTISIM_SYST_GLOBS_FINAL missing MCstat glob')
DIR_MC = str(Path(_mc_glob).parent)
KEYS2LOAD = ['hdr', 'evt']
N_MAX_CONCAT = 999
perTPC_inset = 10

_log(f'loading selected MC from {DIR_MC} ...')
t_mc = time.time()
mc_dfs = dfs_from_dir(
    DIR_MC,
    filename_str='sel_mup-wgts_mcstat',
    keys2load=KEYS2LOAD,
    n_max_concat=N_MAX_CONCAT,
)
mc_evt_df = mc_dfs['evt']
_log(f'  raw mc evt={len(mc_evt_df):,} in {time.time() - t_mc:.1f}s')
mc_evt_df.loc[mc_evt_df.mc.iscc.isna(), ('mc', 'iscc')] = 999

[16:57:24] loading selected MC from /pnfs/sbnd/scratch/users/munjung/cafpyana_out/dfs/2026_05_18_145611__sel_mup-wgts_mcstat/merged_perTPC ...
Found 3 files to process
Files to process: ['/pnfs/sbnd/scratch/users/munjung/cafpyana_out/dfs/2026_05_18_145611__sel_mup-wgts_mcstat/merged_perTPC/2026_05_18_145611__sel_mup-wgts_mcstat_merged_0000.df', '/pnfs/sbnd/scratch/users/munjung/cafpyana_out/dfs/2026_05_18_145611__sel_mup-wgts_mcstat/merged_perTPC/2026_05_18_145611__sel_mup-wgts_mcstat_merged_0001.df', '/pnfs/sbnd/scratch/users/munjung/cafpyana_out/dfs/2026_05_18_145611__sel_mup-wgts_mcstat/merged_perTPC/2026_05_18_145611__sel_mup-wgts_mcstat_merged_0002.df']


  0%|          | 0/3 [00:00<?, ?it/s]

100%|██████████| 3/3 [00:11<00:00,  3.79s/it]

REMEMBER TO RECALCULATE TKI AND CHECK FV!!
[16:57:36]   raw mc evt=74,089 in 11.5s


In [26]:
def perTPC_cut(df):
    in_TPC1 = (
        InFV(df.slc.vertex, det='SBND_TPC1', incathode=perTPC_inset)
        & InFV(df.mu.pfp.trk.end, det='SBND_TPC1', incathode=perTPC_inset)
        & InFV(df.p.pfp.trk.end, det='SBND_TPC1', incathode=perTPC_inset)
    )
    in_TPC2 = (
        InFV(df.slc.vertex, det='SBND_TPC2', incathode=perTPC_inset)
        & InFV(df.mu.pfp.trk.end, det='SBND_TPC2', incathode=perTPC_inset)
        & InFV(df.p.pfp.trk.end, det='SBND_TPC2', incathode=perTPC_inset)
    )
    return in_TPC1 | in_TPC2


def evt_df_fixed(df):
    slc_mudf = df.mu.pfp.trk
    slc_pdf = df.p.pfp.trk
    tki_reco = get_cc1p0pi_tki(
        slc_mudf,
        slc_pdf,
        pad_column_name(('P', 'p_muon'), slc_mudf),
        pad_column_name(('P', 'p_proton'), slc_pdf),
    )
    df['del_Tp_x'] = tki_reco['del_Tp_x']
    df['del_Tp_y'] = tki_reco['del_Tp_y']

    mc_mudf = df.mu.pfp.trk.truth.p
    mc_pdf = df.p.pfp.trk.truth.p
    tki_mc = get_cc1p0pi_tki(
        mc_mudf,
        mc_pdf,
        pad_column_name(('totp',), mc_mudf),
        pad_column_name(('totp',), mc_pdf),
    )
    df['mc_del_Tp_x'] = tki_mc['del_Tp_x']
    df['mc_del_Tp_y'] = tki_mc['del_Tp_y']
    df[('mc', 'del_Tp_x')] = tki_mc['del_Tp_x']
    df[('mc', 'del_Tp_y')] = tki_mc['del_Tp_y']

    n_before = len(df)
    df = df[np.abs(df.slc.vertex.x) > 10]
    if 'topo_categ' not in df.columns:
        df = df.copy()
        df.loc[:, 'topo_categ'] = get_topo_category(df)
    return df, n_before


mc_evt_df = mc_evt_df.loc[perTPC_cut(mc_evt_df)].copy()
for _df in (mc_evt_df,):
    _df[('mu', 'pfp', 'trk', 'phi', '', '', '')] = np.degrees(
        np.arctan2(
            _df[('mu', 'pfp', 'trk', 'dir', 'x', '', '')],
            _df[('mu', 'pfp', 'trk', 'dir', 'y', '', '')],
        )
    )
    _df[('p', 'pfp', 'trk', 'phi', '', '', '')] = np.degrees(
        np.arctan2(
            _df[('p', 'pfp', 'trk', 'dir', 'x', '', '')],
            _df[('p', 'pfp', 'trk', 'dir', 'y', '', '')],
        )
    )

mc_evt_df, n_evt_before_fv = evt_df_fixed(mc_evt_df)
if 'pot_weight' not in mc_evt_df.columns:
    mc_evt_df['pot_weight'] = np.ones(len(mc_evt_df))

cut_cosmic_sel, *_ = get_topo_category(mc_evt_df, ret_cuts=True)
n_cosmic_sel = int(cut_cosmic_sel.sum())
contam_integrated = n_cosmic_sel / len(mc_evt_df) if len(mc_evt_df) else 0.0
_log(
    f'  selected mc evt={len(mc_evt_df):,} (fv cut from {n_evt_before_fv:,}); '
    f'cosmics={n_cosmic_sel} integrated contam={contam_integrated:.4f}'
)
if 'topo_categ' in mc_evt_df.columns:
    _log('  topo_categ counts: ' + mc_evt_df.topo_categ.value_counts().to_dict().__repr__())

[16:57:37]   selected mc evt=74,089 (fv cut from 74,089); cosmics=648 integrated contam=0.0087
[16:57:37]   topo_categ counts: {1: 67660, 2: 2321, 3: 2147, 4: 1114, -1: 648, 0: 199}


In [27]:
cosmics_dict = {}
t_all = time.time()
first_var_done = False

for ivar, var_config in enumerate(tqdm(var_configs, desc='cosmics variables')):
    slug = var_config.var_save_name
    t_var = time.time()
    _log(f'--- [{ivar + 1}/{len(var_configs)}] {slug} ---')
    try:
        pay = process_variable_cosmics(
            offbeam_df,
            intime_df,
            var_config,
            CV_MODE,
            cosmics_disk_dir,
            SAVE_PLOTS,
            flat_uncertainty=COSMICS_FLAT_UNC,
            blow_up_frac_unc_threshold=COSMICS_BLOWUP_FRAC_UNC,
        )
    except Exception as ex:
        _log(f'  SKIP {slug}: {ex}')
        continue

    contam_frac, n_cosmic, n_total = topo_cosmic_contamination_fraction(mc_evt_df, var_config)
    sel_pay = selected_rate_uncertainty_from_cosmics(pay, contam_frac)

    cosmics_dict[slug] = {
        'Cosmics': pay,
        'SelectedRate': {
            'rate': {'cov_frac': sel_pay['cov_frac']},
            'contamination_fraction': sel_pay['contamination_fraction'],
            'n_cosmic': n_cosmic,
            'n_total': n_total,
        },
    }
    fu_tpl = sel_pay['frac_unc_template']
    fu_sel = sel_pay['frac_unc']
    flat_tpl = float(np.max(fu_tpl)) if len(fu_tpl) else 0.0
    _log(
        f'  template flat frac unc={flat_tpl:.4f}; '
        f'contam mean={np.mean(contam_frac):.4f}; '
        f'selected mean frac unc={np.mean(fu_sel):.4f} max={np.max(fu_sel):.4f} '
        f'n_univ={pay.get("n_univ", "?")} in {time.time() - t_var:.1f}s'
    )

    if PLOT_FRAC_UNC_EACH_VAR:
        plot_frac_unc(
            [fu_tpl, fu_sel],
            var_config,
            legends=[f'Cosmic template ({CV_MODE})', 'On selected rate'],
        )
        plt.suptitle(f'{slug}: fractional uncertainty', y=1.02)
        plt.tight_layout()
        plt.show()

    if PLOT_HEATMAP_FIRST_VAR and not first_var_done:
        plot_heatmap(
            pay['rate']['cov_frac'],
            var_config.bins,
            plot_labels=[var_config.var_labels[1], var_config.var_labels[1], 'cov_frac'],
            plot=True,
        )
        plt.suptitle(f'{slug} Cosmics cov_frac')
        plt.tight_layout()
        plt.show()
        first_var_done = True

_log(f'computed {len(cosmics_dict)} variables in {time.time() - t_all:.1f}s')

cosmics variables:   0%|          | 0/18 [00:00<?, ?it/s]

[16:57:37] --- [1/18] integrated ---


[16:57:38]   template flat frac unc=0.3515; contam mean=0.0087; selected mean frac unc=0.0031 max=0.0031 n_univ=1 in 0.7s


/exp/sbnd/app/users/munjung/xsec/freeze/cafpyana/analysis_village/numucc_1p0pi/utils.py:3134: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  def get_text_color(value):
/tmp/ipykernel_854196/3966466977.py:54: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/exp/sbnd/app/users/munjung/xsec/freeze/cafpyana/analysis_village/numucc_1p0pi/utils.py:3232: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  ####
/tmp/ipykernel_854196/3966466977.py:65: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
cosmics variables:   6%|▌         | 1/18 [00:00<00:12,  1.37it/s]

[16:57:38] --- [2/18] muon-p ---
[16:57:38]   template flat frac unc=0.3439; contam mean=0.0073; selected mean frac unc=0.0025 max=0.0063 n_univ=1 in 0.7s


...

[16:57:48] --- [17/18] vertex_y ---
[16:57:49]   template flat frac unc=1.0000; contam mean=0.0094; selected mean frac unc=0.0094 max=0.0279 n_univ=1 in 0.7s


/exp/sbnd/app/users/munjung/xsec/freeze/cafpyana/analysis_village/numucc_1p0pi/utils.py:3134: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  def get_text_color(value):
/tmp/ipykernel_854196/3966466977.py:54: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
cosmics variables:  94%|█████████▍| 17/18 [00:11<00:00,  1.43it/s]

[16:57:49] --- [18/18] vertex_z ---
[16:57:50]   template flat frac unc=1.0000; contam mean=0.0092; selected mean frac unc=0.0092 max=0.0323 n_univ=1 in 0.7s


/exp/sbnd/app/users/munjung/xsec/freeze/cafpyana/analysis_village/numucc_1p0pi/utils.py:3134: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  def get_text_color(value):
/tmp/ipykernel_854196/3966466977.py:54: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
cosmics variables: 100%|██████████| 18/18 [00:12<00:00,  1.43it/s]

[16:57:50] computed 18 variables in 12.6s


In [28]:
if not cosmics_dict:
    raise RuntimeError('no cosmics variables succeeded')

save_cosmics_npz(cosmics_dict, COSMICS_NPZ)
_log(f'wrote {COSMICS_NPZ}')

uncertainty_summary = {}
for slug, cell in cosmics_dict.items():
    tpl_unc = frac_unc_from_pack(cell['Cosmics']['rate'])
    sel_unc = np.sqrt(np.maximum(np.diag(cell['SelectedRate']['rate']['cov_frac']), 0.0))
    cf = np.asarray(cell['SelectedRate']['contamination_fraction'], dtype=float)
    uncertainty_summary[slug] = {
        'cosmic_template_frac_unc': tpl_unc.tolist(),
        'contamination_fraction': cf.tolist(),
        'selected_rate_frac_unc': sel_unc.tolist(),
    }

with open(COSMICS_SUMMARY_JSON, 'w') as f:
    json.dump(
        {
            'schema': 'numucc_cosmics_uncertainty_summary_v1',
            'cv_mode': CV_MODE,
            'mc_dir': DIR_MC,
            'n_selected_mc': int(len(mc_evt_df)),
            'n_cosmic_selected_mc': int(n_cosmic_sel),
            'integrated_contamination_fraction': float(contam_integrated),
            'per_variable': uncertainty_summary,
        },
        f,
        indent=2,
    )
_log('wrote ' + COSMICS_SUMMARY_JSON)

manifest = {
    'schema': 'numucc_cosmics_syst_dict_v2',
    'description': (
        'Cosmic template unisim + contamination-scaled uncertainty on selected MC rate'
    ),
    'cv_mode': CV_MODE,
    'variables': sorted(cosmics_dict.keys()),
    'n_offbeam': int(len(offbeam_df)),
    'n_intime': int(len(intime_df)),
    'n_selected_mc': int(len(mc_evt_df)),
    'n_cosmic_selected_mc': int(n_cosmic_sel),
    'integrated_contamination_fraction': float(contam_integrated),
    'mc_dir': DIR_MC,
    'output_npz': COSMICS_NPZ,
    'uncertainty_summary_json': COSMICS_SUMMARY_JSON,
}
manifest_path = path.join(cosmics_disk_dir, 'cosmics_covariance_manifest.json')
with open(manifest_path, 'w') as f:
    json.dump(manifest, f, indent=2)
_log('wrote ' + manifest_path)

[16:57:50] wrote /exp/sbnd/data/users/munjung/plots/numucc1p0pi/systematics-notebook-cosmic-20260519/Cosmics/cosmics_syst_dict.npz
[16:57:50] wrote /exp/sbnd/data/users/munjung/plots/numucc1p0pi/systematics-notebook-cosmic-20260519/Cosmics/cosmics_uncertainty_summary.json


[16:57:50] wrote /exp/sbnd/data/users/munjung/plots/numucc1p0pi/systematics-notebook-cosmic-20260519/Cosmics/cosmics_covariance_manifest.json


## Optional: merge precomputed chunk pickles

If you ran ``syst_cosmics_chunk.py`` on the grid, set ``CHUNKS_DIR`` and run the cell below
(same as ``get_systematics_cosmics.py aggregate`` / ``syst_cosmics_aggregate.py``).

In [29]:
# CHUNKS_DIR = '/path/to/cosmics_syst-chunked-YYYYMMDD/chunks'
# if CHUNKS_DIR:
#     from argparse import Namespace
#     from analysis_village.numucc_1p0pi.scripts.syst_cosmics_aggregate import run_aggregate
#     run_aggregate(
#         Namespace(
#             chunks_dir=CHUNKS_DIR,
#             syst_disk_root=SYST_DISK_ROOT,
#             out_tag=today_str,
#             error_log=None,
#             no_plots=True,
#             no_save_npz=False,
#             cv_mode=CV_MODE,
#             vars=VAR_NAMES,
#         )
#     )

## Inspect one variable

Offbeam vs intime shapes and sqrt(diag(cov_frac)) for the saved pack.

In [30]:
inspect_var = VariableConfig.tki_del_Tp()
slug = inspect_var.var_save_name
pack = cosmics_dict[slug]['Cosmics']
h_off = np.asarray(pack['univ_offbeam'], dtype=float)
h_in = np.asarray(pack['univ_intime'], dtype=float)
cv = np.asarray(pack['cv_histogram'], dtype=float)

fig, ax = plt.subplots(figsize=(8, 5))
if slug == 'integrated':
    ax.bar(['offbeam', 'intime (scaled)'], [h_off[0], h_in[0]], color=['crimson', 'black'])
    ax.set_ylabel('Events')
else:
    ax.hist(
        inspect_var.bin_centers,
        weights=h_off,
        bins=inspect_var.bins,
        histtype='step',
        color='crimson',
        linewidth=2,
        label='Offbeam data',
    )
    ax.hist(
        inspect_var.bin_centers,
        weights=h_in,
        bins=inspect_var.bins,
        histtype='step',
        color='black',
        linewidth=2,
        label='Intime MC (scaled)',
    )
    ax.hist(
        inspect_var.bin_centers,
        weights=cv,
        bins=inspect_var.bins,
        histtype='step',
        color='tab:blue',
        linestyle='--',
        linewidth=2,
        label=f'CV ({CV_MODE})',
    )
    ax.set_xlim(inspect_var.bins[0], inspect_var.bins[-1])
    ax.set_xlabel(inspect_var.var_labels[0])
    ax.set_ylabel('Events / bin')
ax.legend(frameon=False)
plt.suptitle(f'{slug}: cosmic unisim inputs')
plt.tight_layout()
plt.show()

print(f'{slug} cosmic template sqrt(diag(cov_frac)):', frac_unc_from_pack(pack['rate']))
sel = cosmics_dict[slug]['SelectedRate']
print(f'{slug} contamination fraction:', sel['contamination_fraction'])
print(f'{slug} selected-rate sqrt(diag(cov_frac)):', np.sqrt(np.maximum(np.diag(sel['rate']['cov_frac']), 0.0)))

tki-del_Tp cosmic template sqrt(diag(cov_frac)): [0.61268611 0.61268611 0.61268611 0.61268611 0.61268611 0.61268611
 0.61268611 0.61268611 0.61268611 0.61268611 0.61268611 0.61268611]
tki-del_Tp contamination fraction: [0.00439648 0.00586915 0.00545562 0.00462151 0.00464429 0.01101879
 0.01007423 0.01654887 0.01980743 0.02635741 0.03051181 0.13385827]
tki-del_Tp selected-rate sqrt(diag(cov_frac)): [0.00269366 0.00359595 0.00334258 0.00283154 0.00284549 0.00675106
 0.00617234 0.01013926 0.01213574 0.01614882 0.01869416 0.0820131 ]


/tmp/ipykernel_854196/3656965002.py:47: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [31]:
# Optional: reload NPZ written by this notebook or the aggregate step
# unc = np.load(COSMICS_NPZ, allow_pickle=True)
# slug = VariableConfig.muon_momentum().var_save_name
# pack = unc[slug].item()['Cosmics']
# print('frac unc:', frac_unc_from_pack(pack['rate']))